# Fit de la dynamique générique avec scipy

$$\dot x_i = M_0(x_i) + M_1(x_i) \sum_j A_{ij}\, M_2(x_j)$$

On définit $M_0, M_1, M_2$ à la main comme fonctions Python (degré et forme libre), `scipy.optimize.least_squares` fait le fit non-linéaire. Convention : $\beta_0 = 1$ dans $M_1$ pour lever l'ambiguïté $M_1 \to k M_1, M_2 \to M_2/k$.

In [1]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from functions import load_data, correlation, rmt_clip_correlation

plt.rcParams['figure.dpi'] = 110

# ── Données et adjacence ──────────────────────────────────────────
data = load_data(['stock'], log_returns=True, sort_by_sector=True)
N, T = data.shape[1], data.shape[0]

C_emp   = correlation(data.values, lag=0)
C_clean = rmt_clip_correlation(C_emp, T=T)
A = C_clean.copy()
np.fill_diagonal(A, 0.0)
lam_max = float(np.linalg.eigvalsh((A + A.T) / 2)[-1])
A_norm  = A / lam_max

R = data.values
R0, R1 = R[:-1], R[1:]




In [2]:
# ── Définis tes fonctions ici (un argument nommé par coefficient) ──
def M_0(x, a0, a1, a2):
    return a0 + a1 * x + a2 * x**2

def M_1(x, b1):
    """Convention β_0 = 1 ⇒ pas de paramètre constant."""
    return 1.0 + b1 * x

def M_2(x, c0, c1, c2):
    return c0 + c1 * x + c2 * x**2


# ── Compte automatiquement les paramètres (hors x) ────────────────
def n_params(f):
    return len(inspect.signature(f).parameters) - 1

n_M0, n_M1, n_M2 = n_params(M_0), n_params(M_1), n_params(M_2)


# ── Modèle complet ────────────────────────────────────────────────
def split_params(params):
    return (params[:n_M0],
            params[n_M0:n_M0 + n_M1],
            params[n_M0 + n_M1:])

def model(params, R0, A_norm):
    p0, p1, p2 = split_params(params)
    return M_0(R0, *p0) + M_1(R0, *p1) * (M_2(R0, *p2) @ A_norm.T)

def residuals(params):
    return (model(params, R0, A_norm) - R1).ravel()


# ── Fit ────────────────────────────────────────────────────────────
x0  = np.zeros(n_M0 + n_M1 + n_M2)
res = least_squares(residuals, x0)

p0, p1, p2 = split_params(res.x)

y_hat = model(res.x, R0, A_norm).ravel()
y     = R1.ravel()
r2    = 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()


# ── Résultats ──────────────────────────────────────────────────────
print(f'R² = {r2*100:.5f} %    [{len(res.x)} params, n_iter = {res.nfev}]')
print()
print(f'M_0 params : {p0}')
print(f'M_1 params : {p1}    [β_0 fixé à 1]')
print(f'M_2 params : {p2}')


R² = 0.12630 %    [7 params, n_iter = 10]

M_0 params : [-0.00011567 -0.01839552  0.03503896]
M_1 params : [11.77801841]    [β_0 fixé à 1]
M_2 params : [ 1.79284367e-04 -1.66009602e-02 -7.15943240e-01]


## Split train/test simple — R² OOS

90 % des obs pour le fit (train), les 10 % les plus récents pour le R² out-of-sample. Si la dynamique a changé entre la première période et la dernière, le R² OOS sera plus bas que le R² IS — voire négatif.

In [3]:
# Split simple : 90% train / 10% test (les 10% les plus récents)
TEST_FRAC = 0.10

n = R0.shape[0]
i_split = int(n * (1 - TEST_FRAC))

R0_tr, R1_tr = R0[:i_split], R1[:i_split]
R0_te, R1_te = R0[i_split:], R1[i_split:]

# Fit sur train uniquement
def residuals_tr(params):
    p0, p1, p2 = split_params(params)
    pred = M_0(R0_tr, *p0) + M_1(R0_tr, *p1) * (M_2(R0_tr, *p2) @ A_norm.T)
    return (pred - R1_tr).ravel()

res = least_squares(residuals_tr, np.zeros(n_M0 + n_M1 + n_M2))
p0, p1, p2 = split_params(res.x)


def r2_on(R0_, R1_, params):
    p0_, p1_, p2_ = split_params(params)
    y     = R1_.ravel()
    y_hat = (M_0(R0_, *p0_) + M_1(R0_, *p1_) * (M_2(R0_, *p2_) @ A_norm.T)).ravel()
    return 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()

r2_is  = r2_on(R0_tr, R1_tr, res.x)
r2_oos = r2_on(R0_te, R1_te, res.x)

print(f'n train    : {len(R0_tr)}   ({(1-TEST_FRAC)*100:.0f}%)')
print(f'n test     : {len(R0_te)}   ({TEST_FRAC*100:.0f}%, les plus récents)')
print(f'R² IS  (train) = {r2_is*100:+.5f} %')
print(f'R² OOS (test)  = {r2_oos*100:+.5f} %')
print()
print(f'M_0 params : {p0}')
print(f'M_1 params : {p1}')
print(f'M_2 params : {p2}')


n train    : 12023   (90%)
n test     : 1336   (10%, les plus récents)
R² IS  (train) = +0.14383 %
R² OOS (test)  = +0.19957 %

M_0 params : [-0.00013686 -0.04439159  0.08372004]
M_1 params : [158.03360072]
M_2 params : [ 0.00019495 -0.00864955 -0.07435927]


## Grid search — meilleurs degrés pour $M_0$ et $M_2$

$M_1$ fixé à degré 1 ($\beta_0 = 1, \beta_1$ libre). On balaye $\deg M_0$ et $\deg M_2$ de 0 à 4, on fitte sur train, on sélectionne sur **R² OOS** (test). Si R² OOS chute quand on monte en degré, c'est du sur-ajustement.

DEG_M1_FIXED = 1
DEG_RANGE = range(5)   # 0 à 4 inclus

def fit_poly(d0, d1, d2):
    """Fit avec polynomes (d0, d1, d2). β_0 = 1 dans M_1."""
    n0, n1, n2 = d0 + 1, d1, d2 + 1   # d1 = nombre de params libres dans M_1

    def model(params, X):
        p0 = params[:n0]
        p1 = np.concatenate([[1.0], params[n0:n0 + n1]])
        p2 = params[n0 + n1:]
        return (np.polyval(p0[::-1], X)
                + np.polyval(p1[::-1], X) * (np.polyval(p2[::-1], X) @ A_norm.T))

    def residuals(params):
        return (model(params, R0_tr) - R1_tr).ravel()

    res = least_squares(residuals, np.zeros(n0 + n1 + n2))

    def r2_on(X, Y, params):
        y, yh = Y.ravel(), model(params, X).ravel()
        return 1 - ((y - yh) ** 2).sum() / ((y - y.mean()) ** 2).sum()

    return r2_on(R0_tr, R1_tr, res.x), r2_on(R0_te, R1_te, res.x), len(res.x)


# Grid search
print(f'{"d_M0":>5s} | {"d_M2":>5s} | {"# params":>9s} | {"R² IS (%)":>10s} | {"R² OOS (%)":>11s}')
print('-' * 55)
results = []
for d0 in DEG_RANGE:
    for d2 in DEG_RANGE:
        r2_is, r2_oos, n_p = fit_poly(d0, DEG_M1_FIXED, d2)
        results.append({'d0': d0, 'd2': d2, 'r2_is': r2_is, 'r2_oos': r2_oos, 'n_p': n_p})
        print(f'{d0:>5d} | {d2:>5d} | {n_p:>9d} | {r2_is*100:>9.4f} | {r2_oos*100:>10.4f}')

best = max(results, key=lambda r: r['r2_oos'])
print(f"\nMeilleur OOS : (d_M0={best['d0']}, d_M2={best['d2']}) -> "
      f"R² OOS = {best['r2_oos']*100:.4f}%, R² IS = {best['r2_is']*100:.4f}%")


# Heatmap
import seaborn as sns
n_d = len(list(DEG_RANGE))
mat_oos = np.array([[r for r in results if r['d0']==d0 and r['d2']==d2][0]['r2_oos']*100
                    for d2 in DEG_RANGE for d0 in DEG_RANGE]).reshape(n_d, n_d)
mat_is  = np.array([[r for r in results if r['d0']==d0 and r['d2']==d2][0]['r2_is']*100
                    for d2 in DEG_RANGE for d0 in DEG_RANGE]).reshape(n_d, n_d)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
sns.heatmap(mat_is.T, annot=True, fmt='.3f', cmap='viridis', ax=axes[0],
            xticklabels=list(DEG_RANGE), yticklabels=list(DEG_RANGE))
axes[0].set_xlabel('deg M_2'); axes[0].set_ylabel('deg M_0')
axes[0].set_title('R² IS (%)')

sns.heatmap(mat_oos.T, annot=True, fmt='.3f', cmap='viridis', ax=axes[1],
            xticklabels=list(DEG_RANGE), yticklabels=list(DEG_RANGE))
axes[1].set_xlabel('deg M_2'); axes[1].set_ylabel('deg M_0')
axes[1].set_title(f'R² OOS (%) — meilleur : ({best["d0"]}, {best["d2"]})')
plt.show()